# Molecular dynamics hands-on Python coding practices

This Jupyter Notebook file serves as a gateway to enter the realm of Python programming and molecular dynamics. We'll break down the basic MD loop to several code cells, step-by-step. The main purpose of this coding assignment is to horn your Python programming skills. The first part will be creating multiple arrays, as containers, to store atoms' information such as positions, velocity, acceleration, interatomic relative position vector, forces, etc. The second part will be run time iterations to calculate atomic foreces, accelerations, and update atomic positions. Note that the material parameters are arbitrarily set up. You'll need to use realistic parameters if you want to simulate a real system.

by Hui-Chia Yu, 2025

---

### Part 1: Modify the single phase code and a two phase code

In this practice, we distinguish atoms to A-type and B-type. B-type has a mass twice that of A. Now we have three types of interatomic bonding:
$\varepsilon_{AA}$, $\varepsilon_{AB}$, and $\varepsilon_{BB}$.
In the first test run, we have 
$\varepsilon_{AA} = 0.25$, $\varepsilon_{AB}= 2.5$, and $\varepsilon_{BB}= 0.5.$

What we need is to create a `Mas[i]` array to store different mass for atoms. Let's set the first 40 atoms are A-type, and the rest are B-type. 


In [ ]:
# Let's set up materials parameters, such as atomic mass, epsilon, etc.
# These parameters are set up arbitrarily. For real MD simulations,
# you will need to input realistic materials parameters.

ma = 0.5         # <-- mass
rm = 0.1         # <-- reference radius
# eps = 0.25       # <-- coefficient for energy well
r0 = rm*1.12246  # <-- equilibrium radius

epsAA = 0.25     
epsAB = 2.5
# epsAB = 0.1 
epsBB = 0.5

mb = 1.0

# compuational box size
Lx = 1.
Ly = 1.


# we first import numpy library
import numpy as np
import random

# set up the number of atoms
N = 81

# create position, velocity, and acceleration arrays
Pos = np.zeros((N,2))
Vel = np.random.rand(N, 2)  # <-- random initial velocity
Vel *= 0.001 
Acc = np.random.rand(N, 2)  # <-- random initial acceleration 
Acc *= 0.0001 

# here we set at label < N/2 to be type A, and the rest to be type B
Mas = np.zeros(N)
for at in range(N):
    if at < int(np.floor(N/2)):
        Mas[at] = ma
    else:
        Mas[at] = mb
        

# set up initial atom positions. Let's make a 8x8 configuration
for rw in range(9):           # <-- note that default Python loop goes from 0 to n-1
    for cl in range(9):        # <-- note indent is what Python recognizes nest loop
        Pos[rw*9+cl,0] = cl*r0*1.0 + 0.5*r0 + random.uniform(-0.01, 0.01) # <-- add some randomness
        Pos[rw*9+cl,1] = rw*r0*1.0 + 0.5*r0 + random.uniform(-0.01, 0.01)

    

# save the initial atomic positions
Pos_pr = Pos.copy()
Vel_pr = Vel.copy()
Acc_pr = Acc.copy()



**Let's visualize the initial atom configuration in the cell below.**

In [ ]:
# import matplotlib library, which is built to
# mimic Matlab's plotting functionality

import matplotlib.pyplot as plt  

hfn = int(np.floor(N/2))

# Plot initial atomic configuration 
plt.plot(Pos[:hfn, 0], Pos[:hfn, 1], 'bo')
plt.plot(Pos[hfn:, 0], Pos[hfn:, 1], 'ro')
plt.xlabel("x Position")
plt.ylabel("y Position")
plt.title("Particle Positions")
plt.gca().set_aspect('equal', adjustable='box')
plt.axis([0, Lx, 0, Ly])

When you arrive this part of the code, all the necessary containers should be successfully created.

---
### Part 2: Time stepping

#### Part 2.1

The main difference between the two-phase model and single-phase model is that we will different interaction forces 

$$\vec{f}_{ij} = -\frac{\vec{r}_{ij}}{r_{ij}}  \frac{24 \varepsilon_{AA}}{r_m} \bigg[2 \bigg(\frac{r_m}{r_{ij}}\bigg)^{13} - \bigg(\frac{r_m}{r_{ij}}\bigg)^7 \bigg]$$ 

between A-A atoms.

$$\vec{f}_{ij} = -\frac{\vec{r}_{ij}}{r_{ij}}  \frac{24 \varepsilon_{BB}}{r_m} \bigg[2 \bigg(\frac{r_m}{r_{ij}}\bigg)^{13} - \bigg(\frac{r_m}{r_{ij}}\bigg)^7 \bigg]$$ 

between B-B atoms.

$$\vec{f}_{ij} = -\frac{\vec{r}_{ij}}{r_{ij}}  \frac{24 \varepsilon_{AB}}{r_m} \bigg[2 \bigg(\frac{r_m}{r_{ij}}\bigg)^{13} - \bigg(\frac{r_m}{r_{ij}}\bigg)^7 \bigg]$$ 

between A-B atoms.
So, we can simply set conditions in the previous code when calculate interatomic forces:

$$\text{if}~~at< 40 ~~\text{and}~~ne<40,~~\varepsilon = \varepsilon_{AA}$$

$$\text{if}~~at> 40 ~~\text{and}~~ne>40,~~\varepsilon = \varepsilon_{BB}$$

Other than these two cases:

$$\varepsilon = \varepsilon_{AB}$$


* Copy the single-phase code to here.
* Implement the conditions above to where the interatomic forces are calculated.
* Calculate interatomic forces between pairs. Note we only calculate those atoms
* Calculate total force as before.
* Use Newton's law to calculate acceleration of each atom:
  
$$\vec{a}_i = \frac{\vec{f}_i}{m_i}.$$ 

Note that the two types of atoms have different mass.
* Update the atom positions and impose periodic boundary conditions as before.
    

**Use the code cell to make the time simulation.**

In [ ]:
# import libraries for plotting
from IPython.display import display, clear_output
import time
import matplotlib.pyplot as plt  

# Initialization before the loop
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)


# set up number of time steps and time step size
nstep = 30001
dt = 0.00005
r_cut = 3*r0


# get initial positions from Part 1
Pos = Pos_pr.copy() 
Vel = Vel_pr.copy()
Acc = Acc_pr.copy()

dst = np.zeros((N,N,3))
Fij = np.zeros((N,N,2))
Fi_v = np.zeros((N,2))
Fi_v.fill(0)

tm = 0
cnt = 1
hfn = int(np.floor(N/2))

# Simulation loop
for it in range(0, nstep + 1):


    # Update velocity
    for at in range(N): 
        Vel[at, 0] += Acc[at, 0] * dt
        Vel[at, 1] += Acc[at, 1] * dt   

        
    # Update positions
    for at in range(N): 
        Pos[at,0] += Vel[at,0] * dt
        Pos[at,1] += Vel[at,1] * dt

 
    # Periodic boundary conditions
    for at in range(N):
        # in x direction
        if Pos[at,0] < 0:
            Pos[at,0] = Pos[at,0] + Lx
        if Pos[at,0] > Lx:
            Pos[at,0] = Pos[at,0] - Lx

        # in y direction
        if Pos[at,1] < 0:
            Pos[at,1] = Pos[at,1] + Ly
        if Pos[at,1] > Ly:
            Pos[at,1] = Pos[at,1] - Ly
            
    # Calculate inter-atom distances with periodic boundaries
    for at in range(N - 1):
        for ne in range(at + 1, N):
            dx_c = Pos[ne,0] - Pos[at,0]
            dx_w = dx_c - Lx
            dx_e = dx_c + Lx

            dy_c = Pos[ne,1] - Pos[at,1]
            dy_s = dy_c - Ly
            dy_n = dy_c + Ly

            # X component
            dst[at,ne,0] = min([dx_c, dx_w, dx_e], key=abs)
            dst[ne,at,0] = -dst[at,ne,0]

            # Y component
            dst[at,ne,1] = min([dy_c, dy_s, dy_n], key=abs)
            dst[ne,at,1] = -dst[at,ne,1]

            # Inter-atom distance
            dst[at,ne,2] = np.sqrt(dst[at,ne,0] ** 2 + dst[at,ne,1] ** 2)
            dst[ne,at,2] = dst[at,ne,2]

    # Calculate force
    Fij.fill(0)

    for at in range(N - 1):
        for ne in range(at + 1, N):
            if 0 < dst[at,ne,2] < r_cut:
                
                if at < hfn and ne < hfn:
                    eps = epsAA
                elif at >= hfn and ne >= hfn:
                    esp = epsBB
                else:
                    eps = epsAB
                
                dUdr = 24 * (eps/rm) * (2*(rm/dst[at,ne,2])**13 - (rm/dst[at,ne,2])**7)
 
                Fij[at,ne,0] = -dst[at,ne,0] / dst[at,ne,2] * dUdr
                Fij[at,ne,1] = -dst[at,ne,1] / dst[at,ne,2] * dUdr

                Fij[ne,at,0] = -Fij[at,ne,0]
                Fij[ne,at,1] = -Fij[at,ne,1]

    # Calculate total force of each atom
    for at in range(N):
        Fi_v[at,0] = np.sum(Fij[at,:,0],axis=0)
        Fi_v[at,1] = np.sum(Fij[at,:,1],axis=0)
    
    # calculate acceleration of each atom
    # Acc = Fi_v/ma
    for at in range(N): 
        Acc[at,0] = Fi_v[at,0]/Mas[at]
        Acc[at,1] = Fi_v[at,1]/Mas[at]
        

    # Elapsed time
    tm += dt

    # Visualization
    if it % 100 == 1:
        plt.plot(Pos[:hfn, 0], Pos[:hfn, 1], 'bo')
        plt.plot(Pos[hfn:, 0], Pos[hfn:, 1], 'ro')       
        plt.axis([0, Lx, 0, Ly])
        print(it)
   
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation

---

#### Part 2.2

Now, **let's use Verlet time scheme.**

* Fill the `??` to complete the Verlet method code to run two-phase simulations.
* Try two values of $\epsilon_{AB}$: `2.5` and `0.1`. For $\epsilon_{AB} = 2.5$, A-B bond is stronger than A-A and B-B bonds. You should observe a mixfure of A and B. In contrast, for $\epsilon_{AB} = 0.1$, A-A and B-B bonds are stroner than A-B bond. You should observe a separation between A and B. See the figure below.
<div align="left">
<img src="https://i.ibb.co/DgkFZ3n9/Two-Phase-1.jpg" width="600">
</div>

In [ ]:

# import libraries for plotting
from IPython.display import display, clear_output
import time
import matplotlib.pyplot as plt  

# Initialization before the loop
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)


# set up number of time steps and time step size
nstep = 20001
dt = 0.00005
r_cut = 3*r0
# r_cut = 2.5*rm


# get initial positions from Part 1
Pos = Pos_pr.copy() 
Vel = Vel_pr.copy()
Acc = Acc_pr.copy()

dst = np.zeros((N,N,3))
Fij = np.zeros((N,N,2))
Fi_v = np.zeros((N,2))


# the first time step
Pos_p = Pos_pr.copy()
for at in range(N):
    Pos[at,0] = Pos_p[at,0] + Vel[at,0]*dt + 0.5*Acc[at,0]*dt**2
    
Pos_c = Pos.copy()


tm = 0
cnt = 1
hfn = int(np.floor(N/2))

# Simulation loop
for it in range(1, nstep + 1):


    # Update positions
    for at in range(N): 
        Pos[at,0] = 2*?? - ?? + Acc[at,0]*dt**2
        Pos[at,1] = 2*?? - ?? + Acc[at,1]*dt**2

    Pos_p = ??.copy()
    Pos_c = ??.copy()
    
    # the code below is identical to the Euler-method one
    # Periodic boundary conditions
    for at in range(N):
        # in x direction
        if Pos[at,0] < 0:
            Pos[at,0] = Pos[at,0] + Lx
        if Pos[at,0] > Lx:
            Pos[at,0] = Pos[at,0] - Lx

        # in y direction
        if Pos[at,1] < 0:
            Pos[at,1] = Pos[at,1] + Ly
        if Pos[at,1] > Lx:
            Pos[at,1] = Pos[at,1] - Ly
            
    # Calculate inter-atom distances with periodic boundaries
    for at in range(N - 1):
        for ne in range(at + 1, N):
            dx_c = Pos[ne,0] - Pos[at,0]
            dx_w = dx_c - Lx
            dx_e = dx_c + Lx

            dy_c = Pos[ne,1] - Pos[at,1]
            dy_s = dy_c - Ly
            dy_n = dy_c + Ly

            # X component
            dst[at,ne,0] = min([dx_c, dx_w, dx_e], key=abs)
            dst[ne,at,0] = -dst[at,ne,0]

            # Y component
            dst[at,ne,1] = min([dy_c, dy_s, dy_n], key=abs)
            dst[ne,at,1] = -dst[at,ne,1]

            # Inter-atom distance
            dst[at,ne,2] = np.sqrt(dst[at,ne,0] ** 2 + dst[at,ne,1] ** 2)
            dst[ne,at,2] = dst[at,ne,2]

    # Calculate force
    Fij.fill(0)

    for at in range(N - 1):
        for ne in range(at + 1, N):
            if 0 < dst[at,ne,2] < r_cut:

                if at < hfn and ne < hfn:
                    eps = epsAA
                elif at >= hfn and ne >= hfn:
                    esp = epsBB
                else:
                    eps = epsAB
                    
                dUdr = 24 * (eps/rm) * (2*(rm/dst[at,ne,2])**13 - (rm/dst[at,ne,2])**7)
 
                Fij[at,ne,0] = -dst[at,ne,0] / dst[at,ne,2] * dUdr
                Fij[at,ne,1] = -dst[at,ne,1] / dst[at,ne,2] * dUdr

                Fij[ne,at,0] = -Fij[at,ne,0]
                Fij[ne,at,1] = -Fij[at,ne,1]

    # Calculate total force of each atom
    for at in range(N):
        Fi_v[at,0] = np.sum(Fij[at,:,0],axis=0)
        Fi_v[at,1] = np.sum(Fij[at,:,1],axis=0)
    
    # calculate acceleration of each atom
    # Acc = Fi_v/ma
    for at in range(N): 
        Acc[at,0] = Fi_v[at,0]/ma
        Acc[at,1] = Fi_v[at,1]/ma   
        

    # print(X_Y)

    # Elapsed time
    tm += dt

    # Visualization
    if it % 100 == 1:
        plt.plot(Pos[:hfn, 0], Pos[:hfn, 1], 'bo')
        plt.plot(Pos[hfn:, 0], Pos[hfn:, 1], 'ro')
        plt.axis([0, Lx, 0, Ly])
        print(it)

   
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation

  

### Great! You're done!

Please upload your file to the drop box on the course webpage. Don't forget to add you name on the file name.